# Preprocessing

## Objective

The goal of this notebook is to prepare the cleaned dataset for machine learning. We will spreate features from the target variable, perform feature engineering, identify numerical and categorical variables, split the data into training and testing sets, and build a preprocessing pipeline that can be reused across multiple machine learning models.

##  Import libraries & Load dataset

In [40]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer


pd.set_option('display.max_columns', None)

In [41]:
df = pd.read_csv('../data/processed/customer_churn_clean.csv')

df.head()

,customer_id,gender,senior_citizen,partner,dependents,tenure,phone_service,multiple_lines,internet_service,online_security,online_backup,device_protection,tech_support,streaming_tv,streaming_movies,contract,paperless_billing,payment_method,monthly_charges,total_charges,churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


## Feature Engineering

### Feature 1 --- `total_services`

Business Question:
-   Dose subscribing to more services make customers more loyal?

In [42]:
service_columns = [
    'phone_service',
    'internet_service',
    'multiple_lines',
    'online_security',
    'online_backup',
    'device_protection',
    'tech_support',
    'streaming_tv',
    'streaming_movies'
]

df['total_services'] = (
    df[service_columns].replace(
    {"Yes": 1,
    "No": 0,
    "DSL": 1,
    "Fiber optic": 1,
    "No internet service": 0,
    "No phone service": 0}
    ).sum(axis=1)
).astype(int)

### Feature 2 --- `is_new_customer`

Business Question :

- Are new customer more likely to churn?

In [43]:
df['is_new_customer'] = (df['tenure'] <= 12).astype(int)

### Feature 3 --- `tenure_group`

Business Idea:
- This creates a feature that's more meaningful from a business perspective.

In [44]:
bins = [0, 12, 24, 48, 72]
labels = ['New', 'Developing', 'Established', 'Loyal']


df['tenure_group'] = pd.cut(
    df['tenure'],
    bins=bins,
    labels=labels,
    include_lowest=True
)

### Feature 4 --- `streaming_services`

Business Question

- Do customers who use more entertainment services exhabit different behavior than those who don't?

In [45]:
streaming_cols = [
    'streaming_tv',
    'streaming_movies'
]


df['streaming_services'] = (
    df[streaming_cols]
    .replace({'Yes': 1, 'No': 0, 'No internet service':0})
    .sum(axis=1)
).astype(int)

### Feature 5 --- `has_security_services`

Business Idea:

- Customers using security-related services my be more committed therefore less likely to churn.

In [46]:
security_cols = [
    "online_security",
    "online_backup",
    "device_protection",
    "tech_support"
]

df['has_security_services'] = (
    df[security_cols]
    .replace({'Yes': 1, 'No': 0, 'No internet service': 0})
    .sum(axis=1) > 0
).astype(int)

In [47]:
df.head()

,customer_id,gender,senior_citizen,partner,dependents,tenure,phone_service,multiple_lines,internet_service,online_security,online_backup,device_protection,tech_support,streaming_tv,streaming_movies,contract,paperless_billing,payment_method,monthly_charges,total_charges,churn,total_services,is_new_customer,tenure_group,streaming_services,has_security_services
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No,2,1,New,0,1
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No,4,0,Established,0,1
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes,4,1,New,0,1
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No,4,0,Established,0,1
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes,2,1,New,0,0


In [48]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 26 columns):
 #   Column                 Non-Null Count  Dtype   
---  ------                 --------------  -----   
 0   customer_id            7043 non-null   str     
 1   gender                 7043 non-null   str     
 2   senior_citizen         7043 non-null   int64   
 3   partner                7043 non-null   str     
 4   dependents             7043 non-null   str     
 5   tenure                 7043 non-null   int64   
 6   phone_service          7043 non-null   str     
 7   multiple_lines         7043 non-null   str     
 8   internet_service       7043 non-null   str     
 9   online_security        7043 non-null   str     
 10  online_backup          7043 non-null   str     
 11  device_protection      7043 non-null   str     
 12  tech_support           7043 non-null   str     
 13  streaming_tv           7043 non-null   str     
 14  streaming_movies       7043 non-null   str     
 15

In [49]:
df["total_services"].describe()

count    7043.000000
mean        4.146244
std         2.312720
min         1.000000
25%         2.000000
50%         4.000000
75%         6.000000
max         9.000000
Name: total_services, dtype: float64

In [50]:
df["tenure_group"].value_counts()

tenure_group
Loyal          2239
New            2186
Established    1594
Developing     1024
Name: count, dtype: int64

In [51]:
df["streaming_services"].value_counts()

streaming_services
0    3544
2    1940
1    1559
Name: count, dtype: int64

In [52]:
df["has_security_services"].value_counts()

has_security_services
1    4250
0    2793
Name: count, dtype: int64

## Drop `Customer_id`

The `customer_id` uniquely identifies each customer but contains no predictive information about customer churn.
keeping this feature could introduce unnecessary noise into the model, so it is removed befor training.

In [53]:
df = df.drop(columns=["customer_id"])

## Split Features and Target

In [54]:
# seperating predictor variables and the target variables

X = df.drop(columns=['churn'])
y = df['churn']

## Train/Test split

In [64]:
# spliting the Features(X) and Target(y) in to training and testing sets

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y, 
    test_size=0.2,
    stratify=y, 
    random_state=42
    )

## Categorical Features & Numerical Features

In [56]:
categorical_features = [
    'gender', 'partner', 'dependents', 'phone_service', 'multiple_lines',
    'internet_service', 'online_security', 'online_backup', 'device_protection',
    'tech_support', 'streaming_tv', 'streaming_movies', 'contract',
    'paperless_billing', 'payment_method', 'tenure_group'
]

numerical_features = [
    'senior_citizen', 'tenure', 'monthly_charges', 'total_charges',
    'total_services', 'is_new_customer', 'streaming_services', 'has_security_services'
]

## Bulid the Preprocessing Pipeline

In [57]:
# Pipeline for categorical features
categorical_transformer = Pipeline(steps=[
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# Pipeline for numerical features
numerical_transformer = Pipeline(steps=[
    ('scaler', StandardScaler())
])

# ColumnTransformer combines individual Piplines
preprocessor = ColumnTransformer(transformers=[
    ('num', numerical_transformer, numerical_features),
    ('cat', categorical_transformer, categorical_features)
])

## Validation for Pipeline

In [ ]:
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

In [60]:
X_train_processed.shape

(5634, 53)

In [63]:
X_test_processed.shape

(1409, 53)

# Summary

Here's what we did in this notebook:

1. **Created new features** that could help our model understand customer behavior better:
   - `total_services` — how many services each customer is using.
   - `is_new_customer` — whether the customer joined in the last 12 months or not.
   - `tenure_group` — groups customers into *New*, *Developing*, *Established*, and *Loyal* based on how long they've been with us.
   - `streaming_services` — how many streaming services (TV and movies) they use.
   - `has_security_services` — whether they have any security-related services like online backup or tech support.

2. **Cleaned up and split the data:**
   - Removed `customer_id` since it's just an ID and doesn't help with predictions.
   - Separated the features (X) from what we're trying to predict — `churn` (y).
   - Split everything into training (80%) and testing (20%) sets, making sure both sets have a similar ratio of churned vs non-churned customers.

3. **Organized our features** into two groups:
   - `categorical_features` — things like gender, contract type, payment method, etc.
   - `numerical_features` — numbers like tenure, monthly charges, total charges, and our new count-based features.

4. **Built a preprocessing pipeline** using `ColumnTransformer` so we can apply the right transformations to each group:
   - One-hot encoding for categorical features.
   - Standard scaling for numerical features.

The data is now ready for modeling in the next notebook.

In [66]:
# save the sets to csv 

X_train.to_csv(
    "../data/processed/X_train.csv",
    index=False
)

y_train.to_csv(
    "../data/processed/y_train.csv",
    index=False
)

X_test.to_csv(
    "../data/processed/X_test.csv",
    index=False
)

y_test.to_csv(
    "../data/processed/y_test.csv",
    index=False
)


## Next Step

The dataset now has been successfully processed and is ready for machine learning.

In the next notebook, we will train and compare several classification models to predict customer churn.